<a href="https://colab.research.google.com/github/PierreCr/Digital-Pathology/blob/main/Mini_project_in_Computational_Pathology_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-project in Computational Pathology - Part 2.
# Pierre Cordier, DVM, PhD, Dipl.ECVP.
# 11/10/2026.


In [ ]:
#19.0 Now introduce AI.
from google.colab import drive
drive.mount("/content/drive/")

In [ ]:
#19.1. Now introduce AI.
import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")


from torchvision.models import resnet18, ResNet18_Weights

# Load pretrained ImageNet weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Remove the final 1000-class classification layer
model.fc = torch.nn.Identity()

# Prepare for inference, not training
model.eval()

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Use the preprocessing designed for these weights
preprocess = weights.transforms()

print("Device:", device)
print("Model ready")

In [ ]:
#19.2. Run model on 1 patch.
from google.colab import drive
drive.mount("/content/drive/")

# Load 1 image
patch = Image.open("/content/drive/MyDrive/WSI_CMU-1/normedMacenko_patches512x512/CMU-1_patch_53_normed.png").convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(patch)
plt.axis("off");

# Convert the image to a normalized tensor
image_tensor = preprocess(patch)

# Add a batch dimension: (3, 224, 224) -> (1, 3, 224, 224)
batch = image_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    embedding = model(batch)

embedding = embedding.cpu().numpy()

print("Embedding shape:", embedding.shape)
print("First 10 values:", embedding[0, :10])